# 02 — FOV46 extraction and interactive QC

FOV46 is extracted from the **raw** AnnData/SpatialData. First run extraction, then run the current QC script. The final cells make it easy to inspect and override thresholds before moving to Leiden.

In [ ]:
from pathlib import Path
import sys, os

# Notebook lives in PROJECT_ROOT/jupyter/.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "jupyter" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.executable)
assert (PROJECT_ROOT / "scripts").exists(), "Run this notebook from PROJECT_ROOT/jupyter or PROJECT_ROOT."

In [ ]:
%run ../scripts/04_extract_fov46.py

In [ ]:
%run ../scripts/04a_qc_fov46.py

## Inspect the thresholds actually used

In [ ]:
import json, pandas as pd, scanpy as sc
from IPython.display import display
TAB=PROJECT_ROOT/"results/FOV46/tables"
thr=json.loads((TAB/"recommended_qc_thresholds.json").read_text())
print(json.dumps(thr,indent=2))
display(pd.read_csv(TAB/"qc_threshold_sweep.csv"))

## Inspect QC retention and distributions

In [ ]:
rawq=sc.read_h5ad(PROJECT_ROOT/"results/FOV46/GSM9046088_FOV46_raw_with_qc_flags.h5ad")
print(rawq.obs["qc_pass"].value_counts(dropna=False))
print("Retention:", rawq.obs["qc_pass"].mean())
cols=[c for c in ["total_counts","n_genes_by_counts","control_fraction","Area"] if c in rawq.obs]
display(rawq.obs.groupby("qc_pass")[cols].describe().T)

## Optional threshold override
Edit only if QC inspection supports a change. This cell creates a new QC-filtered object using explicit thresholds, without modifying the production script.

In [ ]:
# Example — uncomment/edit to test an alternative QC set.
# from src.qc import apply_qc
# test = rawq.copy()
# test_thr = dict(thr)
# test_thr["min_counts"] = 75
# test_thr["min_genes"] = 32
# apply_qc(test, test_thr, use_per_fov_flags=False)
# print(test.obs["qc_pass"].value_counts())